In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

# 0) Load & preprocess data 
DATA_DIR = "../data"
OUT_PATH = f"{DATA_DIR}/idealistic_solution_by_zip.csv"

pop = pd.read_csv(f"{DATA_DIR}/population.csv")
inc = pd.read_csv(f"{DATA_DIR}/avg_individual_income.csv")
emp = pd.read_csv(f"{DATA_DIR}/employment_rate.csv")
fac = pd.read_csv(f"{DATA_DIR}/child_care_regulated.csv")

# ZIP to 5-digit strings
def zfix(s):
    s = str(s).split(".")[0]
    return s.zfill(5)

pop["zipcode"]  = pop["zipcode"].map(zfix)
inc["ZIP code"] = inc["ZIP code"].map(zfix)
emp["zipcode"]  = emp["zipcode"].map(zfix)
fac["zip_code"] = fac["zip_code"].map(zfix)

# employment rate to 0–1 if given in %
emp["employment rate"] = emp["employment rate"].astype(float)
if emp["employment rate"].max() > 1.5:
    emp["employment rate"] = emp["employment rate"] / 100.0

# Population (bins as-is; trim 10–14 to 10–12 by 3/5)
# Required in population.csv: '-5', '5-9', '10-14'
assert "-5" in pop.columns
assert "5-9" in pop.columns
assert "10-14" in pop.columns

pop["P_0_5"] = pd.to_numeric(pop["-5"], errors="coerce").fillna(0.0)

pop["P_le12"] = (
    pd.to_numeric(pop["-5"],    errors="coerce").fillna(0.0) +
    pd.to_numeric(pop["5-9"],   errors="coerce").fillna(0.0) +
    0.6 * pd.to_numeric(pop["10-14"], errors="coerce").fillna(0.0)
)

# Join income & employment
dfz = pop[["zipcode","P_0_5","P_le12"]].merge(
    inc.rename(columns={"ZIP code":"zipcode","average income":"avg_income"}),
    on="zipcode", how="left"
).merge(
    emp.rename(columns={"employment rate":"emp_rate"}),
    on="zipcode", how="left"
)

dfz["emp_rate"] = dfz["emp_rate"].fillna(0.0)
dfz["avg_income"] = dfz["avg_income"].fillna(np.inf)
dfz["high_demand"] = (dfz["emp_rate"] >= 0.6) | (dfz["avg_income"] <= 60000)
dfz["r"] = np.where(dfz["high_demand"], 0.5, 1.0/3.0)

# Aggregate existing capacity by ZIP
fac_cap = fac.groupby("zip_code").agg(
    E_tot=("total_capacity","sum"),
    E_05=("infant_capacity","sum")
).reset_index()

tp = fac.groupby("zip_code").agg(
    toddler=("toddler_capacity","sum"),
    preschool=("preschool_capacity","sum")
).reset_index()

fac_cap = fac_cap.merge(tp, on="zip_code", how="left")
fac_cap["E_05"] = fac_cap["E_05"].fillna(0) + fac_cap["toddler"].fillna(0) + fac_cap["preschool"].fillna(0)
fac_cap = fac_cap[["zip_code","E_tot","E_05"]].rename(columns={"zip_code":"zipcode"})

dfz = dfz.merge(fac_cap, on="zipcode", how="left").fillna({"E_tot":0,"E_05":0})

# Facility-level table for expansion caps & baseline trigger
fac_tbl = fac[["facility_id","zip_code","total_capacity"]].copy()

# raw upper bound may be non-integer; slots should be integer → floor
U_raw = np.minimum(1.2*fac_tbl["total_capacity"], 500.0)
fac_tbl["U"] = np.floor(U_raw).astype(int)

fac_tbl["B"] = 20000 + 200*fac_tbl["total_capacity"]   # baseline if >=100% expansion

# 1) Build MILP model
m = gp.Model("NYS_childcare_idealistic")

# Sets & indices
Z = dfz["zipcode"].tolist()
S = ["small","medium","large"]
Ts = {"small":100, "medium":200, "large":400}      # total slots per new facility
K05 = {"small":50,  "medium":100,  "large":200}    # max 0–5 assignable slots per new facility
Cbuild = {"small":65000, "medium":95000, "large":115000}

# Dict params
P05   = dfz.set_index("zipcode")["P_0_5"].to_dict()
Ple12 = dfz.set_index("zipcode")["P_le12"].to_dict()
r     = dfz.set_index("zipcode")["r"].to_dict()
E_tot = dfz.set_index("zipcode")["E_tot"].to_dict()
E05   = dfz.set_index("zipcode")["E_05"].to_dict()

# Facilities per ZIP
F      = fac_tbl["facility_id"].tolist()
zip_of = fac_tbl.set_index("facility_id")["zip_code"].to_dict()
nf     = fac_tbl.set_index("facility_id")["total_capacity"].to_dict()
Ucap   = fac_tbl.set_index("facility_id")["U"].to_dict()
Bfix   = fac_tbl.set_index("facility_id")["B"].to_dict()

# Decision variables (integerize slots)
# number of new facilities (integer)
y = {(z,s): m.addVar(vtype=GRB.INTEGER, lb=0, name=f"y[{z},{s}]") for z in Z for s in S}

# expansion slots at facility f (integer, with integer UB)
x = {f: m.addVar(vtype=GRB.INTEGER, lb=0, ub=int(Ucap[f]), name=f"x[{f}]") for f in F}

# trigger if expansion >= 100% of current capacity
w = {f: m.addVar(vtype=GRB.BINARY, name=f"w[{f}]") for f in F}

# new 0–5 slots by ZIP (integer)
n05_new = {z: m.addVar(vtype=GRB.INTEGER, lb=0, name=f"n05_new[{z}]") for z in Z}

m.update()

# Constraints
# (i) Desert removal per ZIP: total capacity must meet r * P_le12
for z in Z:
    m.addConstr(
        E_tot[z]
        + gp.quicksum(x[f] for f in F if zip_of[f]==z)
        + gp.quicksum(Ts[s]*y[(z,s)] for s in S)
        >= r[z]*Ple12[z],
        name=f"desert[{z}]"
    )

# (ii) U5 coverage per ZIP: existing + new 0–5 >= (2/3)*P_0_5
for z in Z:
    m.addConstr(E05[z] + n05_new[z] >= (2.0/3.0)*P05[z], name=f"u5cov[{z}]")

# (iii) feasible U5 supply from build + expansion
for z in Z:
    m.addConstr(
        n05_new[z] <=
        gp.quicksum(K05[s]*y[(z,s)] for s in S)
        + gp.quicksum(x[f] for f in F if zip_of[f]==z),
        name=f"u5cap_new[{z}]"
    )

# (iv) baseline trigger for >=100% expansion: standard big-M linking
for f in F:
    m.addConstr(x[f] >= int(nf[f]) * w[f], name=f"trigger_lo[{f}]")
    m.addConstr(x[f] <= (int(nf[f]) - 1) * (1 - w[f]) + int(Ucap[f]) * w[f], name=f"trigger_hi[{f}]")


# Objective: Build + Expand + Equipment (0–5)
c_exp = 200.0
obj = gp.quicksum(Cbuild[s]*y[(z,s)] for z in Z for s in S) \
    + gp.quicksum(c_exp*x[f] + Bfix[f]*w[f] for f in F) \
    + 100.0*gp.quicksum(n05_new[z] for z in Z)

m.setObjective(obj, GRB.MINIMIZE)

# Solver controls
m.setParam("TimeLimit", 600)
m.setParam("MIPGap", 0.01)

m.optimize()


# 2) Reporting
if m.status in (GRB.OPTIMAL, GRB.TIME_LIMIT):
    y_sol = {(z,s): int(round(y[(z,s)].X)) for z in Z for s in S if y[(z,s)].X>1e-6}
    x_sol = {f: int(round(x[f].X)) for f in F if x[f].X>1e-6}
    w_sol = {f: int(round(w[f].X)) for f in F if w[f].X>0.5}
    n05_new_sol = {z: int(round(n05_new[z].X)) for z in Z if n05_new[z].X>1e-6}

    total_cost = m.ObjVal
    print(f"Total funding (objective): ${total_cost:,.0f}")

    rows = []
    for z in Z:
        add_x = sum(x_sol[f] for f in x_sol if zip_of[f]==z)
        add_new = sum(Ts[s]*y_sol.get((z,s),0) for s in S)
        supply = int(round(E_tot[z] + add_x + add_new))
        req = r[z]*Ple12[z]
        rows.append({
            "zip": z,
            "new_small": y_sol.get((z,"small"),0),
            "new_medium": y_sol.get((z,"medium"),0),
            "new_large": y_sol.get((z,"large"),0),
            "expansion_slots": add_x,
            "new_u5_slots": n05_new_sol.get(z,0),
            "supply_total": supply,
            "requirement": req,
            "u5_req": (2/3)*P05[z],
            "u5_exist": E05[z]
        })
    pd.DataFrame(rows).sort_values("zip").to_csv(OUT_PATH, index=False)
    print("Saved:", OUT_PATH)
else:
    print("Model status:", m.status)

Set parameter Username
Set parameter LicenseID to value 2705344
Academic license - for non-commercial use only - expires 2026-09-09
Set parameter TimeLimit to value 600
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 11.0.1 build v11.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 36146 rows, 37792 columns and 104575 nonzeros
Model fingerprint: 0xc3c49c22
Variable types: 0 continuous, 37792 integer (15604 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+02]
  Objective range  [1e+02, 2e+05]
  Bounds range     [1e+00, 5e+02]
  RHS range        [1e-01, 1e+04]
Found heuristic solution: objective 5.095365e+08
Presolve removed 35571 rows and 37215 columns
Presolve time: 0.99s
Presolved: 575 rows, 577 columns, 1438 nonzeros
Found heuristic solution: objective 3.032283e+08
Variable types: 0 con

In [ ]:
import pandas as pd, numpy as np, gurobipy as gp
from gurobipy import GRB
from math import radians, sin, cos, asin, sqrt

DATA_DIR = "../data"
zfix=lambda s: str(s).split(".")[0].zfill(5)
def dmi(a,b,c,d):
    R=3958.8; dlat=radians(c-a); dlon=radians(d-b)
    return 2*R*asin((sin(dlat/2)**2+cos(radians(a))*cos(radians(c))*sin(dlon/2)**2)**0.5)

# --- load ---
pop=pd.read_csv(f"{DATA_DIR}/population.csv")
inc=pd.read_csv(f"{DATA_DIR}/avg_individual_income.csv")
emp=pd.read_csv(f"{DATA_DIR}/employment_rate.csv")
fac=pd.read_csv(f"{DATA_DIR}/child_care_regulated.csv")
pot=pd.read_csv(f"{DATA_DIR}/potential_locations.csv")

# --- standardize ZIPs ---
for df,c in [(pop,"zipcode"),(emp,"zipcode"),(pot,"zipcode")]: df[c]=df[c].map(zfix)
fac["zip_code"]=fac["zip_code"].map(zfix)
if "ZIP code" in inc.columns: inc=inc.rename(columns={"ZIP code":"zipcode"})
elif "zipcode" not in inc.columns: raise ValueError("income缺少ZIP")
inc["zipcode"]=inc["zipcode"].map(zfix)
if "avg_income" not in inc.columns:
    if "average income" in inc.columns: inc=inc.rename(columns={"average income":"avg_income"})
    else: inc=inc.rename(columns={ [c for c in inc.columns if c!="zipcode"][0]:"avg_income" })

# --- demand table ---
emp["employment rate"]=pd.to_numeric(emp["employment rate"],errors="coerce")
if emp["employment rate"].max()>1.5: emp["employment rate"]/=100
emp=emp.rename(columns={"employment rate":"emp_rate"})
pop["P_0_5"]=pd.to_numeric(pop.get("-5",0),errors="coerce").fillna(0)
pop["P_le12"]=pop["P_0_5"]+pd.to_numeric(pop.get("5-9",0),errors="coerce").fillna(0)+0.6*pd.to_numeric(pop.get("10-14",0),errors="coerce").fillna(0)
dfz=(pop[["zipcode","P_0_5","P_le12"]]
     .merge(inc[["zipcode","avg_income"]],on="zipcode",how="left")
     .merge(emp[["zipcode","emp_rate"]],on="zipcode",how="left"))
dfz["avg_income"]=pd.to_numeric(dfz["avg_income"],errors="coerce").fillna(np.inf)
dfz["emp_rate"]=dfz["emp_rate"].fillna(0.0)
dfz["r"]=np.where((dfz["emp_rate"]>=0.6)|(dfz["avg_income"]<=60000),0.5,1/3)

# --- existing capacity (ZIP agg) ---
fac[["total_capacity","infant_capacity","toddler_capacity","preschool_capacity"]] = \
    fac[["total_capacity","infant_capacity","toddler_capacity","preschool_capacity"]].apply(
        lambda s: pd.to_numeric(s,errors="coerce").fillna(0.0))
cap=(fac.groupby("zip_code")[["total_capacity","infant_capacity","toddler_capacity","preschool_capacity"]]
     .sum().rename_axis("zipcode").reset_index())
cap["E_tot"]=cap["total_capacity"]
cap["E_05"]=cap["infant_capacity"]+cap["toddler_capacity"]+cap["preschool_capacity"]
dfz=dfz.merge(cap[["zipcode","E_tot","E_05"]],on="zipcode",how="left").fillna({"E_tot":0.0,"E_05":0.0})

# --- facility tables: fac_all (distance) & fac_pos (expandable) ---
fac_all=fac[["facility_id","zip_code","total_capacity","latitude","longitude"]].rename(columns={"zip_code":"zipcode"}).copy()
for c in ["latitude","longitude","total_capacity"]: fac_all[c]=pd.to_numeric(fac_all[c],errors="coerce")
fac_all["nf"]=fac_all["total_capacity"].fillna(0.0).astype(float)
fac_pos=fac_all[fac_all["nf"]>0].copy()

# --- candidate filter (<0.06mi to any existing in same ZIP) + conflicts ---
for c in ["latitude","longitude"]: pot[c]=pd.to_numeric(pot[c],errors="coerce")
pot_use=pot.dropna(subset=["latitude","longitude"]).copy()
rm=[]
for z,g in pot_use.groupby("zipcode"):
    Fz=fac_all[(fac_all["zipcode"]==z)&fac_all["latitude"].notna()&fac_all["longitude"].notna()]
    if Fz.empty: continue
    for i,r in g.iterrows():
        if (Fz.apply(lambda fr:dmi(r.latitude,r.longitude,fr.latitude,fr.longitude),axis=1).min())<0.06: rm.append(i)
pot_use=pot_use.drop(index=list(set(rm)))
C=[]
for z,g in pot_use.groupby("zipcode"):
    ids=g.index.to_list()
    for i in range(len(ids)):
        for j in range(i+1,len(ids)):
            a,b=ids[i],ids[j]
            if dmi(g.at[a,"latitude"],g.at[a,"longitude"],g.at[b,"latitude"],g.at[b,"longitude"])<0.06: C.append((a,b))

# --- sets & params (union-Z; use .get) ---
Ts, K05, Cb={"small":100,"medium":200,"large":400},{"small":50,"medium":100,"large":200},{"small":65000,"medium":95000,"large":115000}
S=["small","medium","large"]
P05=dfz.set_index("zipcode")["P_0_5"].to_dict(); P12=dfz.set_index("zipcode")["P_le12"].to_dict()
rZ=dfz.set_index("zipcode")["r"].to_dict(); E=dfz.set_index("zipcode")["E_tot"].to_dict(); E05=dfz.set_index("zipcode")["E_05"].to_dict()
Z=sorted(set(dfz["zipcode"])|set(pot_use["zipcode"])|set(fac_pos["zipcode"]))
F=fac_pos["facility_id"].tolist(); zipF=fac_pos.set_index("facility_id")["zipcode"].to_dict(); nf=fac_pos.set_index("facility_id")["nf"].to_dict()
L=pot_use.index.tolist(); zipL=pot_use["zipcode"].to_dict()

# --- model ---
m=gp.Model("Task2")
y={(l,s):m.addVar(vtype=GRB.BINARY,name=f"y[{l},{s}]") for l in L for s in S}
x1={f:m.addVar(lb=0,ub=0.1*nf[f],vtype=GRB.CONTINUOUS,name=f"x1[{f}]") for f in F}
x2={f:m.addVar(lb=0,ub=0.05*nf[f],vtype=GRB.CONTINUOUS,name=f"x2[{f}]") for f in F}
x3={f:m.addVar(lb=0,ub=0.05*nf[f],vtype=GRB.CONTINUOUS,name=f"x3[{f}]") for f in F}
n05={z:m.addVar(lb=0,vtype=GRB.CONTINUOUS,name=f"n05[{z}]") for z in Z}

for l in L: m.addConstr(gp.quicksum(y[l,s] for s in S)<=1)
for li,lj in C: m.addConstr(gp.quicksum(y[li,s] for s in S)+gp.quicksum(y[lj,s] for s in S)<=1)
for z in Z:
    m.addConstr(E.get(z,0)+gp.quicksum(x1[f]+x2[f]+x3[f] for f in F if zipF[f]==z)
                +gp.quicksum(Ts[s]*y[l,s] for l in L if zipL[l]==z for s in S) >= rZ.get(z,1/3)*P12.get(z,0))
    m.addConstr(E05.get(z,0)+n05[z] >= (2/3)*P05.get(z,0))
    m.addConstr(n05[z] <= gp.quicksum(K05[s]*y[l,s] for l in L if zipL[l]==z for s in S)
                + gp.quicksum(x1[f]+x2[f]+x3[f] for f in F if zipF[f]==z))

m.setObjective(
    gp.quicksum(Cb[s]*y[l,s] for l in L for s in S)
  + gp.quicksum((20000/nf[f]+200)*x1[f] + (20000/nf[f]+400)*x2[f] + (20000/nf[f]+1000)*x3[f] for f in F)
  + 100*gp.quicksum(n05[z] for z in Z), GRB.MINIMIZE)

m.Params.MIPGap=0.01; m.Params.TimeLimit=1200
m.optimize()

# --- outputs  ---
val=lambda v: float(v.X) if hasattr(v,"X") else np.nan

# 1) selected_sites.csv
pd.DataFrame(
    [{"candidate_idx":l,"zipcode":zipL[l],"size":s,"capacity_add":Ts[s],"u5_add":K05[s],
      "latitude":pot_use.at[l,"latitude"],"longitude":pot_use.at[l,"longitude"]}
     for l in L for s in S if val(y[l,s])>0.5]
).to_csv(f"{DATA_DIR}/selected_sites.csv",index=False)

# 2) by_zip_capacity.csv
rows=[]
for z in Z:
    build=sum(Ts[s]*val(y[l,s]) for l in L if zipL[l]==z for s in S)
    exp  =sum(val(x1[f])+val(x2[f])+val(x3[f]) for f in F if zipF[f]==z)
    tot  =E.get(z,0)+build+exp; thr=rZ.get(z,1/3)*P12.get(z,0)
    rows.append({"zipcode":z,"base_total_capacity":E.get(z,0),
                 "new_build_capacity":build,"expansion_capacity":exp,
                 "total_after":tot,"desert_threshold":thr,
                 "is_desert_after":tot+1e-6<thr})
pd.DataFrame(rows).to_csv(f"{DATA_DIR}/by_zip_capacity.csv",index=False)

print("[done] selected_sites.csv, by_zip_capacity.csv")


Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 1200
Gurobi Optimizer version 11.0.1 build v11.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 298427 rows, 683201 columns and 2476474 nonzeros
Model fingerprint: 0x6f114f21
Variable types: 49403 continuous, 633798 integer (633798 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [2e-01, 9e+01]
  RHS range        [1e-01, 1e+04]
Presolve removed 252929 rows and 543206 columns (presolve time = 24s) ...
Presolve removed 252929 rows and 543206 columns
Presolve time: 24.11s
Presolved: 45498 rows, 139995 columns, 301906 nonzeros
Variable types: 9711 continuous, 130284 integer (130278 binary)
Found heuristic solution: objective 4.602477e+08
Deterministic concurrent LP optimizer: prima

In [ ]:
from math import radians, sin, cos, asin, sqrt
import time
import numpy as np
from scipy.spatial import cKDTree

# Map Task 1 and Task 2 variables to the exact names Task 3 expects
facilities_clean = fac.copy()
facilities_clean = facilities_clean.rename(columns={'total_capacity': 'facility_total_capacity'})

standardize_zip = zfix
potential_locations_df = pot.copy()

# Reconstruct "question1_summary" from the pre-existing dfz table
question1_summary = dfz[['zipcode', 'P_0_5', 'P_le12', 'E_tot', 'E_05']].copy()
question1_summary = question1_summary.rename(columns={
    'P_0_5': 'children_0_5', 
    'P_le12': 'children_0_12',
    'E_tot': 'current_total_slots',
    'E_05': 'current_u5_slots'
})

# Re-calculate high demand
question1_summary['high_demand_area'] = dfz['r'] > 0.35
question1_summary['required_total_slots'] = dfz['r'] * question1_summary['children_0_12']
question1_summary['required_u5_slots'] = (2/3) * question1_summary['children_0_5']



budget_limit = 100_000_000

facility_types = {
    "small": {"total_slots": 100, "u5_slots": 50, "cost": 65000},
    "medium": {"total_slots": 200, "u5_slots": 100, "cost": 95000},
    "large": {"total_slots": 400, "u5_slots": 200, "cost": 115000},
}

# Prepare existing facility geography
import numpy as np
from scipy.spatial import cKDTree

# Convert lat/lon to 3D earth geometry to avoid matrix explosions
def get_cartesian(lat, lon):
    r = 3958.8
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack((r * np.cos(lat) * np.cos(lon), 
                            r * np.cos(lat) * np.sin(lon), 
                            r * np.sin(lat)))

retained_indices = []
for z, g in potential_clean.groupby("zipcode"):
    idx = g.index.values
    coords = facility_coords_by_zip.get(z)
    
    if coords is None or len(coords) == 0:
        retained_indices.extend(idx)
        continue
        
    cand_cart = get_cartesian(g["latitude"].values, g["longitude"].values)
    fac_cart = get_cartesian(coords[:, 0], coords[:, 1])
    
    # Tree easily handles 200,000+ points without crashing RAM
    tree = cKDTree(fac_cart)
    # Count how many existing facilities are within 0.06 miles 
    counts = tree.query_ball_point(cand_cart, r=0.06, return_length=True)
    
    valid = np.array(counts) == 0
    retained_indices.extend(idx[valid])

candidate_locations = potential_clean.loc[retained_indices].copy()
if 'location_id' not in candidate_locations.columns:
    candidate_locations = candidate_locations.reset_index().rename(columns={'index': 'location_id'})
candidate_locations = candidate_locations.sort_index()

if candidate_locations.empty:
    raise RuntimeError("No candidate locations remain after spacing filter.")

conflict_pairs = []
for z, g in candidate_locations.groupby('zipcode'):
    idx = g.index.values
    if len(idx) < 2: continue
        
    cand_cart = get_cartesian(g['latitude'].values, g['longitude'].values)
    tree = cKDTree(cand_cart)
    
    # Automatically finds all conflicting pairs natively (i < j) instantly
    pairs = tree.query_pairs(r=0.06)
    
    for i, j in pairs:
        conflict_pairs.append((idx[i], idx[j]))

# Expansion parameter preparation
facilities_expansion_all = facilities_geo[['facility_id', 'zip_code', 'facility_total_capacity']].copy()
facilities_expansion_all = facilities_expansion_all.dropna(subset=['facility_total_capacity'])
facilities_expansion_all = facilities_expansion_all[facilities_expansion_all['facility_total_capacity'] > 0].copy()

nf = facilities_expansion_all.set_index('facility_id')['facility_total_capacity'].to_dict()
facility_by_zip = facilities_expansion_all.groupby('zip_code')['facility_id'].apply(list).to_dict()

coeff_0_10 = {fid: (20000 + 200 * cap) / cap for fid, cap in nf.items()}
coeff_10_15 = {fid: (20000 + 400 * cap) / cap for fid, cap in nf.items()}
coeff_15_20 = {fid: (20000 + 1000 * cap) / cap for fid, cap in nf.items()}
max_coeff = {fid: max(coeff_0_10[fid], coeff_10_15[fid], coeff_15_20[fid]) for fid in nf}
big_m = {fid: max_coeff[fid] * 1.2 * nf[fid] for fid in nf}

Ts = {t: spec['total_slots'] for t, spec in facility_types.items()}
K05 = {t: spec['u5_slots'] for t, spec in facility_types.items()}
Cbuild = {t: spec['cost'] for t, spec in facility_types.items()}

Z = question1_summary['zipcode'].tolist()
current_total = question1_summary.set_index('zipcode')['current_total_slots'].to_dict()
current_u5 = question1_summary.set_index('zipcode')['current_u5_slots'].to_dict()
required_total = question1_summary.set_index('zipcode')['required_total_slots'].to_dict()
required_u5 = question1_summary.set_index('zipcode')['required_u5_slots'].to_dict()
pop_0_5 = question1_summary.set_index('zipcode')['children_0_5'].to_dict()
pop_0_12 = question1_summary.set_index('zipcode')['children_0_12'].to_dict()
pop_5_12 = {z: max(pop_0_12.get(z, 0) - pop_0_5.get(z, 0), 0) for z in Z}

candidate_index_list = candidate_locations.index.tolist()
regimes = ['seg_0_10', 'seg_10_15', 'seg_15_20']

m_fair = gp.Model("fairness_strict")
m_fair.Params.OutputFlag = 0
m_fair.Params.MIPGap = 0.01
m_fair.Params.TimeLimit = 900
m_fair.Params.DualReductions = 0

y = m_fair.addVars(candidate_index_list, facility_types.keys(), vtype=GRB.BINARY, name="build")
x = m_fair.addVars(nf.keys(), lb=0, ub={fid: 1.2 * cap for fid, cap in nf.items()}, name="expand")
z = m_fair.addVars(nf.keys(), regimes, vtype=GRB.BINARY, name="regime")
cost_expand = m_fair.addVars(nf.keys(), lb=0, name="expand_cost")
n05_new = m_fair.addVars(Z, lb=0, name="u5_new")
gamma_min = m_fair.addVar(lb=0, name="gamma_min")
gamma_max = m_fair.addVar(lb=0, name="gamma_max")

# Expansion regime linking
for fid, cap in nf.items():
    m_fair.addConstr(z[fid, 'seg_0_10'] + z[fid, 'seg_10_15'] + z[fid, 'seg_15_20'] <= 1, name=f"regime_select_{fid}")
    m_fair.addConstr(x[fid] <= 1.2 * cap * (z[fid, 'seg_0_10'] + z[fid, 'seg_10_15'] + z[fid, 'seg_15_20']), name=f"expand_activation_{fid}")
    m_fair.addConstr(x[fid] <= 0.10 * cap + big_m[fid] * (1 - z[fid, 'seg_0_10']), name=f"regime_0_10_ub_{fid}")
    m_fair.addConstr(x[fid] <= 0.15 * cap + big_m[fid] * (1 - z[fid, 'seg_10_15']), name=f"regime_10_15_ub_{fid}")
    m_fair.addConstr(x[fid] <= 1.20 * cap + big_m[fid] * (1 - z[fid, 'seg_15_20']), name=f"regime_15_20_ub_{fid}")
    m_fair.addConstr(x[fid] >= 0.100001 * cap * z[fid, 'seg_10_15'], name=f"regime_10_15_lb_{fid}")
    m_fair.addConstr(x[fid] >= 0.150001 * cap * z[fid, 'seg_15_20'], name=f"regime_15_20_lb_{fid}")
    m_fair.addConstr(cost_expand[fid] <= coeff_0_10[fid] * x[fid] + big_m[fid] * (1 - z[fid, 'seg_0_10']), name=f"cost_ub_0_10_{fid}")
    m_fair.addConstr(cost_expand[fid] >= coeff_0_10[fid] * x[fid] - big_m[fid] * (1 - z[fid, 'seg_0_10']), name=f"cost_lb_0_10_{fid}")
    m_fair.addConstr(cost_expand[fid] <= coeff_10_15[fid] * x[fid] + big_m[fid] * (1 - z[fid, 'seg_10_15']), name=f"cost_ub_10_15_{fid}")
    m_fair.addConstr(cost_expand[fid] >= coeff_10_15[fid] * x[fid] - big_m[fid] * (1 - z[fid, 'seg_10_15']), name=f"cost_lb_10_15_{fid}")
    m_fair.addConstr(cost_expand[fid] <= coeff_15_20[fid] * x[fid] + big_m[fid] * (1 - z[fid, 'seg_15_20']), name=f"cost_ub_15_20_{fid}")
    m_fair.addConstr(cost_expand[fid] >= coeff_15_20[fid] * x[fid] - big_m[fid] * (1 - z[fid, 'seg_15_20']), name=f"cost_lb_15_20_{fid}")

# Candidate site exclusivity and spacing
for idx in candidate_index_list:
    m_fair.addConstr(gp.quicksum(y[idx, t] for t in facility_types) <= 1, name=f"site_limit_{idx}")
for idx_i, idx_j in conflict_pairs:
    m_fair.addConstr(
        gp.quicksum(y[idx_i, t] for t in facility_types) + gp.quicksum(y[idx_j, t] for t in facility_types) <= 1,
        name=f"spacing_{idx_i}_{idx_j}"
    )

candidate_zip = candidate_locations['zipcode'].to_dict()

total_after_expr = {}
u5_after_expr = {}

for z in Z:
    expansion_total = gp.quicksum(x[fid] for fid in facility_by_zip.get(z, []))
    new_total = gp.quicksum(
        Ts[t] * y[idx, t]
        for idx, zip_code in candidate_zip.items() if zip_code == z
        for t in facility_types
    )
    total_after = current_total.get(z, 0) + expansion_total + new_total
    u5_capacity = expansion_total + gp.quicksum(
        K05[t] * y[idx, t]
        for idx, zip_code in candidate_zip.items() if zip_code == z
        for t in facility_types
    )
    u5_after = current_u5.get(z, 0) + n05_new[z]

    total_after_expr[z] = total_after
    u5_after_expr[z] = u5_after

    m_fair.addConstr(total_after >= required_total.get(z, 0), name=f"total_cover_{z}")
    m_fair.addConstr(n05_new[z] <= u5_capacity, name=f"u5_cap_{z}")
    m_fair.addConstr(u5_after >= required_u5.get(z, 0), name=f"u5_cover_{z}")
    m_fair.addConstr(total_after >= u5_after, name=f"u5_le_total_{z}")

    pop_total = pop_0_12.get(z, 0)
    if pop_total > 0:
        m_fair.addConstr(total_after >= gamma_min * pop_total, name=f"fair_lower_{z}")
        m_fair.addConstr(total_after <= gamma_max * pop_total, name=f"fair_upper_{z}")

m_fair.addConstr(gamma_max - gamma_min <= 0.1, name="fair_band")

expansion_cost_terms = gp.quicksum(cost_expand[fid] for fid in nf)
construction_cost_terms = gp.quicksum(Cbuild[t] * y[idx, t] for idx in candidate_index_list for t in facility_types)
equipment_cost_terms = 100 * gp.quicksum(n05_new[z] for z in Z)

total_cost_expr = expansion_cost_terms + construction_cost_terms + equipment_cost_terms
m_fair.addConstr(total_cost_expr <= budget_limit, name="budget_cap")

coverage_terms = []
for z in Z:
    pop_u5_val = pop_0_5.get(z, 0)
    pop_5_12_val = pop_5_12.get(z, 0)
    u5_after = u5_after_expr[z]
    total_after = total_after_expr[z]
    school_age_after = total_after - u5_after
    if pop_u5_val > 0:
        coverage_terms.append((2 / 3) * (1 / pop_u5_val) * u5_after)
    if pop_5_12_val > 0:
        coverage_terms.append((1 / 3) * (1 / pop_5_12_val) * school_age_after)

m_fair.setObjective(gp.quicksum(coverage_terms), GRB.MAXIMIZE)

solve_start = time.time()
m_fair.optimize()
solve_time = time.time() - solve_start

status_code = m_fair.Status
print(f"Solve status: {status_code}")
if status_code == GRB.INFEASIBLE:
    m_fair.computeIIS()
    infeasible = [c.ConstrName for c in m_fair.getConstrs() if c.IISConstr]
    print(f"Model infeasible. IIS size: {len(infeasible)}")
else:
    if status_code in (GRB.OPTIMAL, GRB.TIME_LIMIT) and m_fair.SolCount:
        print(f"Objective value (social coverage index): {m_fair.ObjVal:.6f}")
        print(f"Solve time: {solve_time:,.2f} seconds")
        print(f"Gamma min / max: {gamma_min.X:.4f} / {gamma_max.X:.4f}")
        print(f"Band width: {gamma_max.X - gamma_min.X:.4f}")
        total_cost_value = total_cost_expr.getValue()
        print(f"Total cost: ${total_cost_value:,.0f} (budget ${budget_limit:,.0f})")
    else:
        print("Model ended without an incumbent solution.")

candidate_selection_fair = []
if status_code in (GRB.OPTIMAL, GRB.TIME_LIMIT) and m_fair.SolCount:
    for idx in candidate_index_list:
        for t in facility_types:
            if y[idx, t].X > 0.5:
                row = candidate_locations.loc[idx]
                candidate_selection_fair.append({
                    'candidate_index': idx,
                    'location_id': row['location_id'],
                    'zipcode': row['zipcode'],
                    'facility_type': t,
                    'total_slots': Ts[t],
                    'u5_slots': K05[t],
                    'latitude': row['latitude'],
                    'longitude': row['longitude']
                })

expansion_records_fair = []
if status_code in (GRB.OPTIMAL, GRB.TIME_LIMIT) and m_fair.SolCount:
    for fid in nf:
        added = x[fid].X
        if added > 1e-6:
            regs = {r: z[fid, r].X for r in regimes}
            chosen = max(regs, key=regs.get)
            expansion_records_fair.append({
                'facility_id': fid,
                'zipcode': facilities_expansion_all.loc[facilities_expansion_all['facility_id'] == fid, 'zip_code'].iloc[0],
                'added_slots': added,
                'regime': chosen,
                'regime_cost_per_slot': {
                    'seg_0_10': coeff_0_10[fid],
                    'seg_10_15': coeff_10_15[fid],
                    'seg_15_20': coeff_15_20[fid]
                }[chosen]
            })

fair_solution_summary = None
if status_code in (GRB.OPTIMAL, GRB.TIME_LIMIT) and m_fair.SolCount:
    fair_solution_summary = question1_summary[['zipcode', 'high_demand_area', 'children_0_12', 'children_0_5', 'current_total_slots', 'current_u5_slots', 'required_total_slots', 'required_u5_slots']].copy()
    fair_solution_summary['expansion_slots'] = fair_solution_summary['zipcode'].map({fid_zip: sum(x[fid].X for fid in facility_by_zip.get(fid_zip, [])) for fid_zip in Z})
    fair_solution_summary['new_build_slots'] = fair_solution_summary['zipcode'].map({
        z: sum(Ts[t] for idx, zip_code in candidate_zip.items() if zip_code == z for t in facility_types if y[idx, t].X > 0.5)
        for z in Z
    })
    fair_solution_summary['u5_new_slots'] = fair_solution_summary['zipcode'].map({z: n05_new[z].X for z in Z})
    fair_solution_summary[['expansion_slots', 'new_build_slots', 'u5_new_slots']] = fair_solution_summary[['expansion_slots', 'new_build_slots', 'u5_new_slots']].fillna(0)
    fair_solution_summary['total_after'] = fair_solution_summary['current_total_slots'] + fair_solution_summary['expansion_slots'] + fair_solution_summary['new_build_slots']
    fair_solution_summary['u5_after'] = fair_solution_summary['current_u5_slots'] + fair_solution_summary['u5_new_slots']
    fair_solution_summary['school_age_after'] = (fair_solution_summary['total_after'] - fair_solution_summary['u5_after']).clip(lower=0)
    fair_solution_summary['total_ratio'] = np.where(
        fair_solution_summary['children_0_12'] > 0,
        fair_solution_summary['total_after'] / fair_solution_summary['children_0_12'],
        np.nan
    )
    fair_solution_summary['u5_ratio'] = np.where(
        fair_solution_summary['children_0_5'] > 0,
        fair_solution_summary['u5_after'] / fair_solution_summary['children_0_5'],
        np.nan
    )
    fair_solution_summary['school_age_ratio'] = np.where(
        (fair_solution_summary['children_0_12'] - fair_solution_summary['children_0_5']) > 0,
        fair_solution_summary['school_age_after'] / (fair_solution_summary['children_0_12'] - fair_solution_summary['children_0_5']),
        np.nan
    )
    fair_solution_summary['is_desert_after'] = fair_solution_summary['total_after'] + 1e-5 < fair_solution_summary['required_total_slots']
    remaining_deserts = int(fair_solution_summary['is_desert_after'].sum())
    print(f"Remaining deserts: {remaining_deserts}")
    total_gap = fair_solution_summary['total_ratio'].dropna()
    if not total_gap.empty:
        print(f"Coverage range: {total_gap.min():.4f} - {total_gap.max():.4f}")

candidate_selection_fair_df = pd.DataFrame(candidate_selection_fair)
expansion_fair_df = pd.DataFrame(expansion_records_fair)



Solve status: 3
Model infeasible. IIS size: 1
